# Metricas para CEQ

Esse algoritmo pega atributos numéricos de uma base, calcula índices estatísticos de qualidade e gráficos de controle apenas para os exemplos da **classe escolhida pelo usuário** (0 ou 1, perguntada logo no início da célula abaixo), e gera rankings de importância dos atributos de acordo com essas métricas, salvando cada ranking em Excel.

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import MinMaxScaler

# 1. Carregar arquivo
df = pd.read_excel("amostra.xlsx")
df.columns = df.columns.astype(str).str.strip()

# ❓ 1.1 Classe a ser avaliada e nome do cenário -- perguntados UMA ÚNICA VEZ, aqui,
# e reutilizados por TODO o notebook (Bloco 1 CEQ, descritivas, SVD/Laplacian/SPEC).
classes_disponiveis = df.iloc[:, -1].unique().tolist()
classe_desejada = input(f"Classes disponíveis: {classes_disponiveis}\nDigite a classe desejada: ")
nome_cenario = input("Nome do cenário/base (ex.: SECOM, Cenário A, Cenário B...): ")

# 2. Separar atributos (X) e classe resposta (y)
X = df.iloc[:, :-1].reset_index(drop=True)
y = df.iloc[:, -1].reset_index(drop=True)

# 3. Selecionar atributos numéricos
X_quant = X.select_dtypes(include=[np.number]).copy()

# 3.1 Imputação defensiva por mediana -- este notebook NÃO depende mais de o
# notebook 1 já ter imputado amostra.xlsx antes. Se já vier imputado, isto é um
# no-op (nenhum NaN restante); se vier cru, imputa aqui mesmo.
n_ausentes_x = int(X_quant.isna().sum().sum())
if n_ausentes_x > 0:
    X_quant = X_quant.fillna(X_quant.median())
print(f"Imputação defensiva (X_quant): {n_ausentes_x} valor(es) ausente(s) preenchido(s).")

# 📐 4. Normalização dos dados
X_norm = pd.DataFrame(MinMaxScaler().fit_transform(X_quant), columns=X_quant.columns)

# 🧪 5. Métricas CEQ

# 📏 Limites de especificação de Cp/Cpk/Pp/Ppk
# Os fornecedores dos dados (SECOM, Caminhões) não têm limites de especificação de
# engenharia. Na ausência deles, LSL e USL são estimados pelos PERCENTIS 1 e 99 de
# cada sensor na classe escolhida: a faixa que contém 98% das leituras, ignorando
# 1% dos valores mais extremos de cada lado.
# Por que não mínimo/máximo (versão anterior): uma única leitura atípica mudava o
# limite; o ranking mudava com o tamanho da amostra e não era reprodutível (top-20
# do Pp com 9% de concordância entre duas metades da amostra, contra 57% com P1/P99);
# e Pp = (máx-mín)/6σ era EXATAMENTE AmplitudeSigma/6 (mesmo ranking do controle).
# Por que não os quartis/cercas do boxplot: tornam os 4 índices praticamente iguais
# ao IQRSigma do grupo de controle (ρ de Spearman 0,89 a 1,00).
# O notebook 1 usa os MESMOS percentis na regra de parada -- mude os dois juntos.
PERCENTIL_LSL = 1
PERCENTIL_USL = 99
def sigma_curto_prazo_mr(dados):
    """Estima o sigma de CURTO PRAZO (variação inerente, dentro de subgrupo) via
    amplitude móvel (moving range) entre observações consecutivas -- a técnica
    padrão de um gráfico de indivíduos (I-MR) quando não há subgrupos racionais
    definidos (Montgomery, 2020). d2=1.128 é a constante para subgrupo de tamanho 2.
    PREMISSA: assume que a ordem das linhas em df_classe reflete a ordem real de
    coleta/processo -- se os dados não estiverem em ordem temporal, este sigma
    perde sua interpretação de 'curto prazo'.
    """
    valores = dados.values
    if len(valores) < 2:
        return np.nan
    mr = np.abs(np.diff(valores))
    mr_bar = np.mean(mr)
    d2 = 1.128
    return mr_bar / d2


def calcular_ceq(df_classe):
    """Cp/Cpk usam sigma de CURTO prazo (moving range) -- 'capacidade potencial'.
    Pp/Ppk usam sigma de LONGO prazo (desvio-padrão amostral geral) -- 'desempenho
    real'. Diferente de antes, as quatro fórmulas NÃO são mais idênticas entre si:
    a diferença Cp vs. Pp (e Cpk vs. Ppk) é justamente o que permite avaliar
    estabilidade -- Cp ≈ Pp indica processo estável; Cp >> Pp indica variação extra
    de longo prazo (deriva, instabilidade) não capturada pela variação ponto-a-ponto.
    """
    resultados = []
    for col in df_classe.columns:
        dados = df_classe[col].dropna()
        LSL, USL = np.percentile(dados, [PERCENTIL_LSL, PERCENTIL_USL]) if len(dados) else (np.nan, np.nan)
        media = dados.mean()

        std_longo = np.std(dados, ddof=0)          # usado por Pp/Ppk
        std_curto = sigma_curto_prazo_mr(dados)     # usado por Cp/Cpk

        invalido = (
            len(dados) < 2 or std_longo == 0
            or std_curto == 0 or pd.isna(std_curto)
        )
        if invalido:
            # Índice indefinido (sensor constante): NaN -> fim do ranking. Antes era 0,
            # o que o deixava ACIMA de sensores com Cpk/Ppk negativo (média fora de [P1, P99]).
            valores = {m: np.nan for m in ["Cp", "Cpk", "Pp", "Ppk"]}
        else:
            Cp = (USL - LSL) / (6 * std_curto)
            Cpu = (USL - media) / (3 * std_curto)
            Cpl = (media - LSL) / (3 * std_curto)
            Cpk = min(Cpu, Cpl)
            Pp = (USL - LSL) / (6 * std_longo)
            Ppu = (USL - media) / (3 * std_longo)
            Ppl = (media - LSL) / (3 * std_longo)
            Ppk = min(Ppu, Ppl)
            valores = {"Cp": Cp, "Cpk": Cpk, "Pp": Pp, "Ppk": Ppk}

        resultados.append({"Feature": col, **valores})
    return pd.DataFrame(resultados)

# 📉 6. Métricas dos gráficos de controle
#
# p/np (gráfico de UNIDADES não-conformes) e c/u (gráfico de DEFEITOS) só são
# conceitos distintos quando "unidade não-conforme" ≠ "defeito" -- ou seja,
# quando uma mesma amostra pode acumular vários defeitos (em vários atributos
# diferentes) ao mesmo tempo. Por isso:
#   - p/np olham SÓ para este atributo isolado: entre as n amostras, quantas
#     ficaram fora de ±1σ NESTE atributo.
#   - c/u olham para o processo como um todo: entre as amostras em que ESTE
#     atributo ficou fora de ±1σ, quantos atributos (TODOS, não só este)
#     também estavam fora de controle na MESMA amostra, simultaneamente. Um
#     atributo cujos desvios tendem a coincidir com desvios em vários outros
#     atributos (instabilidade generalizada do processo) recebe c/u mais alto
#     que um atributo cujos desvios ocorrem isolados -- diferença que p/np,
#     por definição, não capturam.
# Sem essa distinção, as quatro fórmulas colapsavam aos pares (p≡u, np≡c),
# já que todas reaproveitavam a mesma contagem univariada.
def calcular_graficos_controle(df_classe):
    n = len(df_classe)
    colunas = df_classe.columns

    # Matriz binária de "defeito": 1 onde o valor sai de ±1σ da média da
    # PRÓPRIA coluna -- calculada de uma vez para TODOS os atributos, pois
    # c/u (abaixo) precisam saber quantos atributos estão fora de controle
    # simultaneamente em cada amostra, não só o atributo atual.
    matriz_defeito = pd.DataFrame(0, index=df_classe.index, columns=colunas, dtype=int)
    for col in colunas:
        dados_col = df_classe[col]
        dados_validos = dados_col.dropna()
        media = dados_validos.mean()
        std = np.std(dados_validos, ddof=0)  # consistente com o ddof=0 de calcular_ceq(), acima
        if len(dados_validos) >= 2 and std > 0:
            fora_de_controle = (dados_col < media - std) | (dados_col > media + std)
            matriz_defeito[col] = fora_de_controle.fillna(False).astype(int)

    # D_i: nº TOTAL de atributos fora de controle, simultaneamente, na amostra i
    # (conceito clássico de "defeitos por unidade" -- uma unidade pode ter
    # vários defeitos ao mesmo tempo, em características/atributos diferentes).
    defeitos_simultaneos_por_amostra = matriz_defeito.sum(axis=1)

    resultados = []
    for col in colunas:
        dados = df_classe[col].dropna()
        std = np.std(dados, ddof=0)

        if len(dados) < 2 or std == 0:
            valores = {m: np.nan for m in ["p", "np", "c", "u"]}   # indefinido -> fim do ranking
        else:
            amostras_defeituosas_nesta_coluna = matriz_defeito[col].astype(bool)
            n_defeituosas = int(amostras_defeituosas_nesta_coluna.sum())

            p = n_defeituosas / n
            np_val = n_defeituosas
            c = int(defeitos_simultaneos_por_amostra[amostras_defeituosas_nesta_coluna].sum())
            u = c / n

            valores = {"p": p, "np": np_val, "c": c, "u": u}

        resultados.append({"Feature": col, **valores})
    return pd.DataFrame(resultados)

# ⏱ Lista para registrar tempos
tempos_execucao = []

# 7. Função para salvar ranking com medição de tempo
def salvar_ranking(df, metrica, normalizado):
    inicio = time.time()  # ⏱ início

    df_rank = df[["Feature", metrica]].rename(columns={metrica: "Score"})
    df_rank = df_rank.sort_values(by="Score", ascending=False, na_position='last').reset_index(drop=True)
    df_rank["Ranking"] = df_rank.index + 1
    nome = f"Ranking_{metrica}_{'Normalizado' if normalizado else 'Original'}.xlsx"
    df_rank.to_excel(nome, index=False)

    fim = time.time()  # ⏱ fim
    duracao = fim - inicio

    # Armazena no log
    tempos_execucao.append((nome, duracao))
    print(f"Ranking salvo em: {nome} | Tempo: {duracao:.4f} segundos")

# 🔁 8. Executar para dados originais e normalizados
for normalizado, base in zip([False, True], [X_quant, X_norm]):
    # Filtra pela classe ESCOLHIDA (antes estava fixo em y == 1)
    base_classe1 = base[y.astype(str) == str(classe_desejada)].reset_index(drop=True)

    # 🧪 CEQ
    df_ceq = calcular_ceq(base_classe1)
    for metrica in ["Cp", "Cpk", "Pp", "Ppk"]:
        salvar_ranking(df_ceq, metrica, normalizado)

    # 📉 Gráficos de controle
    df_gc = calcular_graficos_controle(base_classe1)
    for metrica in ["p", "np", "c", "u"]:
        salvar_ranking(df_gc, metrica, normalizado)

# 9. Salvar log de tempos em Excel
df_tempos = pd.DataFrame(tempos_execucao, columns=["Arquivo", "Tempo (s)"])
df_tempos.to_excel("Tempos_Execucao.xlsx", index=False)
print("Log de tempos salvo em: Tempos_Execucao.xlsx")


🩹 Imputação defensiva (X_quant): 0 valor(es) ausente(s) preenchido(s).
✅ Ranking salvo em: Ranking_Cp_Original.xlsx | Tempo: 0.0776 segundos
✅ Ranking salvo em: Ranking_Cpk_Original.xlsx | Tempo: 0.1477 segundos
✅ Ranking salvo em: Ranking_Pp_Original.xlsx | Tempo: 0.0707 segundos
✅ Ranking salvo em: Ranking_Ppk_Original.xlsx | Tempo: 0.0771 segundos
✅ Ranking salvo em: Ranking_p_Original.xlsx | Tempo: 0.0660 segundos
✅ Ranking salvo em: Ranking_np_Original.xlsx | Tempo: 0.0476 segundos
✅ Ranking salvo em: Ranking_c_Original.xlsx | Tempo: 0.0496 segundos
✅ Ranking salvo em: Ranking_u_Original.xlsx | Tempo: 0.0500 segundos
✅ Ranking salvo em: Ranking_Cp_Normalizado.xlsx | Tempo: 0.0489 segundos
✅ Ranking salvo em: Ranking_Cpk_Normalizado.xlsx | Tempo: 0.0419 segundos
✅ Ranking salvo em: Ranking_Pp_Normalizado.xlsx | Tempo: 0.0534 segundos
✅ Ranking salvo em: Ranking_Ppk_Normalizado.xlsx | Tempo: 0.0471 segundos
✅ Ranking salvo em: Ranking_p_Normalizado.xlsx | Tempo: 0.0595 segundos
✅ Ra

# Algoritmos Comuns univariáveis

Abaixo o texto que trás algoritmos e medidas de referência para escolher atributos

https://www.sciencedirect.com/science/article/pii/S2772662223000802

Carrega o arquivo no dataframe e manda escolher qual a classe resposta. Depois utiliza essa informação para produtizar os rankings

### Escolha de classe

In [2]:
!pip install pandas openpyxl skrebate --q

In [3]:
import pandas as pd
import numpy as np

# 📥 Ler o arquivo Excel
df = pd.read_excel('amostra.xlsx')
df.columns = df.columns.astype(str).str.strip()

# ❓ classe_desejada e nome_cenario já foram perguntados na célula do Bloco 1 (acima)
# e são reutilizados aqui -- o usuário responde uma única vez para o notebook todo.

# 0) Sensores SEM NENHUMA leitura válida em TODA a amostra (todas as
# classes) -- nenhuma mediana (nem global, nem por classe) resolve isso, então
# removemos essas colunas da análise como um todo, de forma explícita e
# registrada (em vez de deixar NaN residual quebrar SVD Entropy/Laplacian/SPEC
# mais à frente). Isso é esperado em amostras pequenas; na base completa
# (589 linhas) tende a não ocorrer, mas o notebook fica protegido de qualquer
# forma.
colunas_numericas_df = df.select_dtypes(include=[np.number]).columns
colunas_totalmente_vazias = colunas_numericas_df[df[colunas_numericas_df].isna().all()]
if len(colunas_totalmente_vazias) > 0:
    df = df.drop(columns=colunas_totalmente_vazias)
    print(f"{len(colunas_totalmente_vazias)} sensor(es) sem NENHUMA leitura válida em toda a "
          f"amostra foram removidos da análise: {list(colunas_totalmente_vazias)}")

# Base FILTRADA (somente a classe escolhida)
# Usada pelos métodos intrínsecos que não comparam contra y (SVD Entropy,
# Laplacian Score, SPEC Score) -- eles medem a estrutura/estabilidade dos
# próprios dados dentro da classe escolhida, no mesmo espírito das métricas
# CEQ do Bloco 1.
df_classe = df[df.iloc[:, -1].astype(str) == str(classe_desejada)].reset_index(drop=True)

# Imputação defensiva por mediana em df_classe -- este notebook não depende
# mais de o notebook 1 já ter imputado amostra.xlsx antes (se já vier imputado,
# isto é um no-op). Sem isso, SVD Entropy/Laplacian/SPEC e as 4 estatísticas
# descritivas (mais abaixo) herdariam NaNs silenciosamente se a ordem de execução
# dos notebooks fosse invertida ou pulada.
#
# Caso especial: quando a classe escolhida tem poucas amostras (ex.: classe
# de anomalia rara), é possível que ALGUM sensor não tenha NENHUMA leitura
# válida DENTRO dessa classe, mesmo tendo leitura em outras classes -- nesse
# caso a mediana calculada só com a classe fica ela mesma NaN, e o fillna()
# não preenche nada. Para essas colunas, usamos como aproximação a mediana
# GLOBAL (calculada com TODAS as classes, já sem as colunas 100% vazias
# removidas acima -- por isso a mediana global nunca fica NaN aqui).
colunas_numericas_classe = df_classe.select_dtypes(include=[np.number]).columns
mediana_classe = df_classe[colunas_numericas_classe].median()
mediana_global = df[colunas_numericas_classe].median()  # todas as classes, não só a escolhida
colunas_sem_dado_na_classe = mediana_classe[mediana_classe.isna()].index

n_ausentes_classe = int(df_classe[colunas_numericas_classe].isna().sum().sum())
if n_ausentes_classe > 0:
    mediana_para_preencher = mediana_classe.fillna(mediana_global)
    df_classe[colunas_numericas_classe] = df_classe[colunas_numericas_classe].fillna(mediana_para_preencher)

n_restantes = int(df_classe[colunas_numericas_classe].isna().sum().sum())
print(f"Imputação defensiva (df_classe): {n_ausentes_classe} valor(es) ausente(s) preenchido(s) "
      f"({len(colunas_sem_dado_na_classe)} coluna(s) sem nenhum valor válido na classe, mas com "
      f"dado em outras classes -- usada a mediana global nessas).")
assert n_restantes == 0, (
    f"Ainda restam {n_restantes} valor(es) ausente(s) após a imputação -- isso não deveria "
    f"acontecer, pois as colunas 100% vazias já foram removidas acima. Investigar."
)

# Arquivo só com as linhas da classe escolhida (já imputadas), para conferência
arquivo_classe = f"amostra_classe_{classe_desejada}.xlsx"
df_classe.to_excel(arquivo_classe, index=False)
print(f"Linhas da classe {classe_desejada} salvas em '{arquivo_classe}' ({len(df_classe)} linhas)")

# 🔁 Base BINÁRIA: classe escolhida (1) vs. todas as outras (0)
# Usada pelos métodos supervisionados (correlações, ANOVA, Chi², T-Score, KS,
# Hellinger etc.), que precisam de duas classes para ter o que comparar.
y_bin = (df.iloc[:, -1].astype(str) == str(classe_desejada)).astype(int).reset_index(drop=True)

# 🔢 Atributos numéricos da base completa, com ausentes preenchidos pela
# mediana (evita erros em Chi², Mutual Information, correlações etc.)
#
# A coluna-alvo (última coluna, ex.: "Pass/Fail") é removida ANTES do
# select_dtypes -- como ela normalmente é codificada como número (-1/1, 0/1
# etc.), sem essa remoção ela entra como se fosse mais um atributo e acaba
# sendo "ranqueada" junto dos sensores de verdade pelos métodos supervisionados
# (Pearson, ANOVA, Chi², ...), com correlação perfeita consigo mesma (vazamento
# do rótulo para dentro do próprio ranking de atributos).
coluna_alvo = df.columns[-1]
X_full = df.drop(columns=[coluna_alvo]).select_dtypes(include=[np.number]).copy()
X_full = X_full.fillna(X_full.median())

# 🚫 Sensores CONSTANTES na amostra (mesmo valor em todas as linhas, de todas as
# classes): não carregam informação nenhuma para nenhum método. Antes, cada
# método os tratava de um jeito -- alguns os colocavam no FIM (NaN), mas o
# Laplacian Score (score 0 = "melhor") e o Gini (impureza 0 por um bug do qcut)
# os colocavam no TOPO, e o Information Gain dava a eles MI positivo por ruído
# do estimador. Agora a regra é única: montar_ranking() força Score = NaN para
# eles, e todos os rankings os colocam nas últimas posições.
CONSTANTES = set(X_full.columns[X_full.std(ddof=0) == 0])
print(f"🚫 {len(CONSTANTES)} sensor(es) constante(s) na amostra -- vão para o fim de todos os rankings.")

# Atributos da classe escolhida SEM a coluna-alvo (antes, SVD Entropy,
# Laplacian e SPEC usavam df_classe.select_dtypes(...) e o alvo 'Pass/Fail'
# entrava como se fosse um sensor).
X_classe = df_classe.drop(columns=[coluna_alvo]).select_dtypes(include=[np.number])

print(f"Classe escolhida: {classe_desejada}")
print(f"Amostras na classe escolhida: {(y_bin == 1).sum()} | Amostras nas demais: {(y_bin == 0).sum()}")
print(f"Amostras em df_classe (só a classe escolhida): {len(df_classe)}")

# --- Tamanho amostral da classe de anomalia, salvo em aba própria ---
# dentro do próprio amostra.xlsx (junto das abas 'Amostra' e 'Resultados' já
# geradas pelo notebook 1) -- não cria um arquivo novo, só acrescenta uma aba.
MAX_AMOSTRAS_NOTEBOOK1 = 500  # mesmo teto usado no notebook 1 (amostragem adaptativa)
info_amostral_code10 = pd.DataFrame([{
    "Base/Cenário": nome_cenario,
    "n_total_amostra": len(df),
    "n_classe_anomalia": int((y_bin == 1).sum()),
    "atingiu_teto_500": len(df) >= MAX_AMOSTRAS_NOTEBOOK1,
    # O notebook 1 usa df.sample(..., ) / df_restante.sample(...) SEM replace=True
    # em nenhum ponto -- na implementação atual, reamostragem com reposição
    # nunca ocorre. Fixo em False; revisar aqui se o notebook 1 mudar.
    "reamostragem_com_reposicao": False,
}])

with pd.ExcelWriter("amostra.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    info_amostral_code10.to_excel(writer, sheet_name="Info_Amostral_CODE10", index=False)
print(f"Tamanho amostral salvo na aba 'Info_Amostral_CODE10' de amostra.xlsx")


🩹 Imputação defensiva (df_classe): 0 valor(es) ausente(s) preenchido(s) (0 coluna(s) sem nenhum valor válido na classe, mas com dado em outras classes -- usada a mediana global nessas).
💾 Linhas da classe -1 salvas em 'amostra_classe_-1.xlsx' (731 linhas)
🚫 122 sensor(es) constante(s) na amostra -- vão para o fim de todos os rankings.
Classe escolhida: -1
Amostras na classe escolhida: 731 | Amostras nas demais: 52
Amostras em df_classe (só a classe escolhida): 731
📋 CODE-10: tamanho amostral salvo na aba 'Info_Amostral_CODE10' de amostra.xlsx


### Funções auxiliares\n\n`montar_ranking` padroniza TODA métrica em 3 colunas -- `Feature`, `Score`, `Ranking` -- já ordenadas do melhor (Ranking 1) para o pior atributo, de acordo com a convenção de cada métrica (`ascending=True` para métricas em que menor é melhor, como Laplacian Score e Gini Index; `ascending=False` para as demais). `salvar_ranking` grava o Excel e registra o tempo de execução no log central, evitando repetir esse bloco em cada célula.

In [4]:
import time
import os
import numpy as np
import pandas as pd

def montar_ranking(features, scores, ascending=False):
    """Padroniza a saída em 3 colunas: Feature, Score, Ranking (1 = melhor atributo)."""
    ranking_df = pd.DataFrame({"Feature": list(features), "Score": np.asarray(scores, dtype=float)})
    # Sensores constantes (sem informação) sempre no fim -- ver CONSTANTES no bloco de preparo
    ranking_df.loc[ranking_df["Feature"].isin(globals().get("CONSTANTES", set())), "Score"] = np.nan
    ranking_df = ranking_df.sort_values(by="Score", ascending=ascending, na_position="last").reset_index(drop=True)
    ranking_df["Ranking"] = ranking_df.index + 1
    return ranking_df


def registrar_tempo(nome_arquivo, tempo_execucao, arquivo_tempos="Tempos_Execucao.xlsx"):
    """Adiciona uma linha ao log de tempos de execução, criando o arquivo se necessário."""
    novo_registro = pd.DataFrame([[nome_arquivo, tempo_execucao]], columns=["Arquivo", "Tempo (s)"])

    if os.path.exists(arquivo_tempos):
        try:
            df_tempos = pd.read_excel(arquivo_tempos)
            df_tempos = pd.concat([df_tempos, novo_registro], ignore_index=True)
            with pd.ExcelWriter(arquivo_tempos, engine="openpyxl", mode="w") as writer:
                df_tempos.to_excel(writer, index=False)
            print(f"Tempo registrado em '{arquivo_tempos}'")
        except PermissionError:
            temp_file = "Tempos_Execucao_temp.xlsx"
            with pd.ExcelWriter(temp_file, engine="openpyxl", mode="w") as writer:
                novo_registro.to_excel(writer, index=False)
            print(f"'{arquivo_tempos}' está aberto no Excel. Registro salvo em '{temp_file}'.")
    else:
        with pd.ExcelWriter(arquivo_tempos, engine="openpyxl", mode="w") as writer:
            novo_registro.to_excel(writer, index=False)
        print(f"Criado novo arquivo de tempos: '{arquivo_tempos}'")


def salvar_ranking(ranking_df, nome_arquivo, tempo_execucao, mensagem=None):
    """Salva o ranking (já no formato Feature/Score/Ranking) em Excel e registra o tempo."""
    ranking_df.to_excel(nome_arquivo, index=False)
    print(mensagem or f"Ranking salvo em '{nome_arquivo}'")
    print(f"⏱ Tempo de execução: {tempo_execucao:.4f} segundos")
    registrar_tempo(nome_arquivo, tempo_execucao)


### Estatísticas descritivas: grupo de controle do CEQ

Duas versões de cada uma das 4 estatísticas (Amplitude/σ, IQR/σ, Assimetria, Curtose):

- **`Ranking_<Nome>.xlsx`** -- calculada só sobre a classe escolhida (`df_classe`), com a mesma população do CEQ (Bloco 1). É o grupo de controle válido para o teste de equivalência (Friedman/Nemenyi, notebook 3).
- **`Ranking_<Nome>_BaseCompleta.xlsx`** -- forma convencional de estatística descritiva, sobre a base inteira (todas as classes). Serve de referência/comparação, mas NÃO é um controle pareado com o CEQ, pois usa outra população.

*Cp/Cpk usam σ de curto prazo (amplitude móvel) e Pp/Ppk σ de longo prazo; estas descritivas usam o σ de longo prazo (`ddof=0`).*

*Maior valor = mais instável/discrepante (mesma convenção do restante do notebook). A coluna de classe (rótulo) é excluída do cálculo.*

In [5]:
from scipy.stats import skew, kurtosis

inicio = time.time()

# 🔢 Somente a classe escolhida (já imputada de forma defensiva acima) -- mesma
# lógica intrínseca de SVD Entropy / Laplacian / SPEC
# (a coluna de rótulo, a última, fica de fora -- não é atributo)
rotulo_col = df.columns[-1]
df_numerico_comp = df_classe.drop(columns=[rotulo_col], errors="ignore").select_dtypes(include=[np.number])
# Base completa (todas as classes), imputada, sem o rótulo -- versão convencional
df_numerico_completo = X_full.drop(columns=[rotulo_col], errors="ignore")


def amplitude_sigma(coluna):
    """(max-min)/sigma_geral -- estatística descritiva simples de amplitude,
    usando o MESMO sigma de longo prazo que Pp/Ppk (não o de curto prazo de
    Cp/Cpk) -- é o comparador correto para a crítica original do revisor."""
    x = coluna.dropna().values
    if len(x) < 2:
        return np.nan
    std = np.std(x, ddof=0)
    return (np.max(x) - np.min(x)) / std if std > 0 else np.nan


def iqr_sigma(coluna):
    """(Q3-Q1)/sigma -- versão robusta a outliers da amplitude normalizada."""
    x = coluna.dropna().values
    if len(x) < 2:
        return np.nan
    std = np.std(x, ddof=0)
    q1, q3 = np.percentile(x, [25, 75])
    return (q3 - q1) / std if std > 0 else np.nan


def assimetria_abs(coluna):
    """|skewness| -- comparador de Cpk (deslocamento em relação ao centro)."""
    x = coluna.dropna().values
    if len(x) < 3:
        return np.nan
    return abs(skew(x, bias=True))


def curtose_abs(coluna):
    """|kurtosis| (excesso, em relação à normal) -- comparador de p (peso de cauda)."""
    x = coluna.dropna().values
    if len(x) < 3:
        return np.nan
    return abs(kurtosis(x, fisher=True, bias=True))


metricas_comparacao = {
    "AmplitudeSigma": amplitude_sigma,
    "IQRSigma": iqr_sigma,
    "Assimetria": assimetria_abs,
    "Curtose": curtose_abs,
}

versoes = [
    ("",              df_numerico_comp),      # classe escolhida (controle pareado com o CEQ)
    ("_BaseCompleta", df_numerico_completo),  # base inteira (uso convencional)
]
for nome_metrica, func in metricas_comparacao.items():
    for sufixo, base_calc in versoes:
        inicio_m = time.time()
        scores = [func(base_calc[col]) for col in base_calc.columns]
        ranking_df = montar_ranking(base_calc.columns, scores, ascending=False)
        tempo_execucao = time.time() - inicio_m
        nome_arq = f"Ranking_{nome_metrica}{sufixo}.xlsx"
        salvar_ranking(ranking_df, nome_arq, tempo_execucao,
                       f"Ranking {nome_metrica}{sufixo} salvo em '{nome_arq}'")

tempo_execucao_total = time.time() - inicio
print(f"⏱ Tempo total do bloco de comparação: {tempo_execucao_total:.4f} segundos")

✅ Ranking AmplitudeSigma salvo em 'Ranking_AmplitudeSigma.xlsx'
⏱ Tempo de execução: 0.0674 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'
✅ Ranking AmplitudeSigma_BaseCompleta salvo em 'Ranking_AmplitudeSigma_BaseCompleta.xlsx'
⏱ Tempo de execução: 0.0407 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'
✅ Ranking IQRSigma salvo em 'Ranking_IQRSigma.xlsx'
⏱ Tempo de execução: 0.0757 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'
✅ Ranking IQRSigma_BaseCompleta salvo em 'Ranking_IQRSigma_BaseCompleta.xlsx'
⏱ Tempo de execução: 0.0759 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_34084\921038577.py:40: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return abs(skew(x, bias=True))


✅ Ranking Assimetria salvo em 'Ranking_Assimetria.xlsx'
⏱ Tempo de execução: 0.2435 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_34084\921038577.py:40: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return abs(skew(x, bias=True))


✅ Ranking Assimetria_BaseCompleta salvo em 'Ranking_Assimetria_BaseCompleta.xlsx'
⏱ Tempo de execução: 0.2486 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_34084\921038577.py:48: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return abs(kurtosis(x, fisher=True, bias=True))


✅ Ranking Curtose salvo em 'Ranking_Curtose.xlsx'
⏱ Tempo de execução: 0.2484 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_34084\921038577.py:48: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return abs(kurtosis(x, fisher=True, bias=True))


✅ Ranking Curtose_BaseCompleta salvo em 'Ranking_Curtose_BaseCompleta.xlsx'
⏱ Tempo de execução: 0.2659 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'
⏱ Tempo total do bloco de comparação: 1.9814 segundos


### ANOVA F-value manual\n\n*Maior F-score = melhor atributo (mais separação entre as classes).*

In [6]:
import numpy as np

inicio = time.time()

# 🔢 X: atributos numéricos da base completa | y: classe escolhida vs. resto
X = X_full.values
y = y_bin.values

def anova_f_score(X, y):
    scores = []
    classes = np.unique(y)
    overall_means = np.mean(X, axis=0)

    for i in range(X.shape[1]):
        between = 0.0
        within = 0.0
        for c in classes:
            X_c = X[y == c, i]
            mean_c = np.mean(X_c)
            between += len(X_c) * (mean_c - overall_means[i]) ** 2
            within += np.sum((X_c - mean_c) ** 2)
        score = between / within if within != 0 else 0
        scores.append(score)
    return scores

f_scores = anova_f_score(X, y)

ranking_df = montar_ranking(X_full.columns, f_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_ANOVA.xlsx", tempo_execucao)


✅ Ranking salvo em 'Ranking_ANOVA.xlsx'
⏱ Tempo de execução: 0.0270 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


### SVD Entropy\n\n*Maior entropia = melhor atributo (mais informação/variabilidade estrutural).*

In [7]:
import numpy as np

inicio = time.time()

# 🔢 Somente a classe escolhida, sem a coluna-alvo (métrica intrínseca, não usa y)
#
# CORREÇÃO: a versão anterior calculava a SVD de CADA COLUNA ISOLADA. Um vetor
# tem um único valor singular, então P = [1] e a entropia é SEMPRE 0 -- o
# "ranking" era só a ordem das colunas na planilha. A forma da literatura
# (Varshavsky et al., 2006, Bioinformatics) é a CONTRIBUIÇÃO À ENTROPIA de cada
# atributo: CE_i = E(todos) - E(todos menos i), onde E é a entropia
# normalizada dos valores singulares ao quadrado da matriz (padronizada).
# Maior CE = atributo que mais contribui para a estrutura dos dados.
def entropia_svd(autovalores):
    v = np.clip(autovalores, 0, None)
    v = v[v > 1e-12]
    p = v / v.sum()
    return -np.sum(p * np.log(p)) / np.log(len(p)) if len(p) > 1 else 0.0

variaveis = [c for c in X_classe.columns if X_classe[c].std(ddof=0) > 0]
Z = X_classe[variaveis].to_numpy(dtype=float)
Z = (Z - Z.mean(0)) / Z.std(0)
G = Z.T @ Z                     # valores singulares² de Z = autovalores de ZᵀZ
E_total = entropia_svd(np.linalg.eigvalsh(G))
ce = {}
for i, col in enumerate(variaveis):
    manter = np.r_[0:i, i + 1:len(variaveis)]
    ce[col] = E_total - entropia_svd(np.linalg.eigvalsh(G[np.ix_(manter, manter)]))

scores = [ce.get(col, np.nan) for col in X_classe.columns]   # constantes -> NaN (fim)
ranking_df = montar_ranking(X_classe.columns, scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_SVDEntropy.xlsx", tempo_execucao,
               "Ranking SVD Entropy salvo em 'Ranking_SVDEntropy.xlsx'")


✅ Ranking SVD Entropy salvo em 'Ranking_SVDEntropy.xlsx'
⏱ Tempo de execução: 22.6806 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


### Laplacian Score\n\n*Menor score = melhor atributo (mais discriminativo na vizinhança local).*

In [8]:
from scipy.spatial.distance import pdist, squareform

inicio = time.time()

# 🔢 Somente a classe escolhida, SEM a coluna-alvo (métrica intrínseca, não usa y).
# CORREÇÃO: antes usava df_classe.select_dtypes(...), que incluía 'Pass/Fail' como
# candidato; e atributos constantes recebiam score 0 = "melhor" (menor é melhor
# aqui). Agora o alvo fica de fora e os constantes vão para o fim (CONSTANTES).
X = X_classe.values
atributos = X_classe.columns

def laplacian_score(X, k=5):
    n_samples = X.shape[0]

    dist_matrix = squareform(pdist(X, metric='euclidean'))
    W = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        idx = np.argsort(dist_matrix[i])[1:k + 1]
        W[i, idx] = 1
        W[idx, i] = 1

    D = np.diag(W.sum(axis=1))
    L = D - W

    scores = []
    for i in range(X.shape[1]):
        f = X[:, i]
        f = f - np.mean(f)
        numerator = f.T @ L @ f
        denominator = f.T @ D @ f + 1e-12
        scores.append(numerator / denominator)
    return scores

scores = laplacian_score(X)

ranking_df = montar_ranking(atributos, scores, ascending=True)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_LaplacianScore.xlsx", tempo_execucao,
               "Ranking Laplacian Score salvo em 'Ranking_LaplacianScore.xlsx'")


✅ Ranking Laplacian Score salvo em 'Ranking_LaplacianScore.xlsx'
⏱ Tempo de execução: 0.4260 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


### SPEC Score\n\n*Maior score = melhor atributo.*

In [9]:
from scipy.linalg import eigh

inicio = time.time()

# 🔢 Somente a classe escolhida, SEM a coluna-alvo (antes 'Pass/Fail' entrava como candidato)
X = X_classe.values
atributos = X_classe.columns

def spec_score(X, k=5, n_eig=5):
    n_samples = X.shape[0]

    dist_matrix = squareform(pdist(X, metric='euclidean'))
    W = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        idx = np.argsort(dist_matrix[i])[1:k + 1]
        W[i, idx] = 1
        W[idx, i] = 1

    D = np.diag(W.sum(axis=1))
    L = D - W

    eigvals, eigvecs = eigh(L)
    eigvecs_selected = eigvecs[:, 1:n_eig + 1]

    scores = []
    for i in range(X.shape[1]):
        f = X[:, i]
        f = (f - np.mean(f)) / (np.std(f) + 1e-12)
        proj = eigvecs_selected.T @ f
        scores.append(np.sum(proj ** 2))
    return scores

scores = spec_score(X)

ranking_df = montar_ranking(atributos, scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_SPEC.xlsx", tempo_execucao,
               "Ranking SPEC salvo em 'Ranking_SPEC.xlsx'")


✅ Ranking SPEC salvo em 'Ranking_SPEC.xlsx'
⏱ Tempo de execução: 2.7991 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## ReliefF\n\n*Maior score = melhor atributo. (As duas implementações antigas de ReliefF, que sobrescreviam o mesmo arquivo, foram unificadas nesta única célula usando a biblioteca `skrebate`.)*

In [10]:
from sklearn.preprocessing import MinMaxScaler
from skrebate import ReliefF

inicio = time.time()

# 🔄 Normalizar os dados (base completa, classe escolhida vs. resto)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_full)

relieff = ReliefF(n_neighbors=10, n_features_to_select=X_full.shape[1])
relieff.fit(X_scaled, y_bin.values)

ranking_df = montar_ranking(X_full.columns, relieff.feature_importances_, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_ReliefF.xlsx", tempo_execucao,
               "Ranking ReliefF salvo em 'Ranking_ReliefF.xlsx'")


✅ Ranking ReliefF salvo em 'Ranking_ReliefF.xlsx'
⏱ Tempo de execução: 10.0639 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


### Algoritmos univariados

Se encaixam no rankeamento

## Correlação de Pearson\n\n*Maior |r| = melhor atributo.*

In [11]:
inicio = time.time()

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_full)

correlacoes = []
for i, col in enumerate(X_full.columns):
    r = np.corrcoef(X_scaled[:, i], y_bin.values)[0, 1]
    correlacoes.append(abs(r))

ranking_df = montar_ranking(X_full.columns, correlacoes, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_Pearson.xlsx", tempo_execucao,
               "Ranking Pearson r salvo em 'Ranking_Pearson.xlsx'")


✅ Ranking Pearson r salvo em 'Ranking_Pearson.xlsx'
⏱ Tempo de execução: 0.0770 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


c:\Users\USUARIO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\USUARIO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## Correlação de Spearman ρ\n\n*Maior |ρ| = melhor atributo.*

In [12]:
from scipy.stats import spearmanr

inicio = time.time()

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_full)

correlacoes = []
for i, col in enumerate(X_full.columns):
    rho, _ = spearmanr(X_scaled[:, i], y_bin.values)
    correlacoes.append(abs(rho))

ranking_df = montar_ranking(X_full.columns, correlacoes, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_Spearman.xlsx", tempo_execucao,
               "Ranking Spearman ρ salvo em 'Ranking_Spearman.xlsx'")


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_34084\2048594994.py:10: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(X_scaled[:, i], y_bin.values)


✅ Ranking Spearman ρ salvo em 'Ranking_Spearman.xlsx'
⏱ Tempo de execução: 0.2133 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Correlação de Kendall τ\n\n*Maior |τ| = melhor atributo.*

In [13]:
from scipy.stats import kendalltau

inicio = time.time()

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_full)

correlacoes = []
for i, col in enumerate(X_full.columns):
    tau, _ = kendalltau(X_scaled[:, i], y_bin.values)
    correlacoes.append(abs(tau))

ranking_df = montar_ranking(X_full.columns, correlacoes, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_Kendall.xlsx", tempo_execucao,
               "Ranking Kendall τ salvo em 'Ranking_Kendall.xlsx'")


✅ Ranking Kendall τ salvo em 'Ranking_Kendall.xlsx'
⏱ Tempo de execução: 0.2062 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Qui-quadrado (χ²)\n\n*Maior estatística = melhor atributo. (p-value não é mais salvo, para manter as 3 colunas padronizadas -- se precisar dele, ele ainda é calculado na célula.)*

In [14]:
from sklearn.feature_selection import chi2

inicio = time.time()

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_full)  # chi2 exige valores não negativos

chi2_scores, p_values = chi2(X_scaled, y_bin.values)

ranking_df = montar_ranking(X_full.columns, chi2_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_Chi2.xlsx", tempo_execucao,
               "Ranking Qui-quadrado salvo em 'Ranking_Chi2.xlsx'")


✅ Ranking Qui-quadrado salvo em 'Ranking_Chi2.xlsx'
⏱ Tempo de execução: 0.0288 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## mutual_info_classif\n\n*Maior informação mútua = melhor atributo.*

In [15]:
from sklearn.feature_selection import mutual_info_classif

inicio = time.time()

info_gain_scores = mutual_info_classif(X_full, y_bin.values, discrete_features=False, random_state=42)  # determinístico

ranking_df = montar_ranking(X_full.columns, info_gain_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_InformationGain.xlsx", tempo_execucao,
               "Ranking Ganho de Informação salvo em 'Ranking_InformationGain.xlsx'")


✅ Ranking Ganho de Informação salvo em 'Ranking_InformationGain.xlsx'
⏱ Tempo de execução: 1.5891 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Gain Ratio\n\n*Maior razão de ganho = melhor atributo. (Information_Gain e Intrinsic_Value não são mais salvos, para manter as 3 colunas padronizadas.)*

In [16]:
inicio = time.time()

info_gain_scores = mutual_info_classif(X_full, y_bin.values, discrete_features=False, random_state=42)  # determinístico

def intrinsic_value(column):
    bins = pd.qcut(column, q=10, duplicates='drop')
    probs = bins.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs + 1e-9))

intrinsic_values = [intrinsic_value(X_full[col]) for col in X_full.columns]
gain_ratios = [ig / iv if iv != 0 else 0 for ig, iv in zip(info_gain_scores, intrinsic_values)]

ranking_df = montar_ranking(X_full.columns, gain_ratios, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_GainRatio.xlsx", tempo_execucao,
               "Ranking Razão de Ganho salvo em 'Ranking_GainRatio.xlsx'")


✅ Ranking Razão de Ganho salvo em 'Ranking_GainRatio.xlsx'
⏱ Tempo de execução: 2.4888 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## F-Score (ANOVA via sklearn)\n\n*Maior F-score = melhor atributo. (p-value não é mais salvo, para manter as 3 colunas padronizadas.)*

In [17]:
from sklearn.feature_selection import f_classif

inicio = time.time()

f_scores, p_values = f_classif(X_full, y_bin.values)

ranking_df = montar_ranking(X_full.columns, f_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_FScore_ANOVA.xlsx", tempo_execucao,
               "Ranking F-Score (ANOVA) salvo em 'Ranking_FScore_ANOVA.xlsx'")


✅ Ranking F-Score (ANOVA) salvo em 'Ranking_FScore_ANOVA.xlsx'
⏱ Tempo de execução: 0.0344 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


c:\Users\USUARIO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [  5  13  42  49  52  69  74  97 141 149 178 179 186 189 190 191 192 193
 194 206 209 226 229 230 231 232 233 234 235 236 237 240 241 242 243 256
 257 258 259 260 261 262 263 264 265 266 276 284 313 314 315 322 325 326
 327 328 329 330 342 347 364 369 370 371 372 373 374 375 378 379 380 381
 394 395 396 397 398 399 400 401 402 403 404 414 422 449 450 451 458 461
 462 463 464 465 466 478 481 498 501 502 503 504 505 506 507 508 509 512
 513 514 515 528 529 530 531 532 533 534 535 536 537 538] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\USUARIO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


## T-Score (teste t)\n\n*Maior |t| = melhor atributo (Score = |estatística t|; p-value não é mais salvo).*

In [18]:
from scipy.stats import ttest_ind

inicio = time.time()

classes = np.unique(y_bin.values)  # sempre [0, 1] com a base binária

t_scores = []
for col in X_full.columns:
    group1 = X_full[y_bin.values == classes[0]][col]
    group2 = X_full[y_bin.values == classes[1]][col]
    t_stat, p_val = ttest_ind(group1, group2, equal_var=False)
    t_scores.append(abs(t_stat))

ranking_df = montar_ranking(X_full.columns, t_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_TScore.xlsx", tempo_execucao,
               "Ranking T-Score salvo em 'Ranking_TScore.xlsx'")


c:\Users\USUARIO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


✅ Ranking T-Score salvo em 'Ranking_TScore.xlsx'
⏱ Tempo de execução: 6.3567 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Fisher Score\n\n*Maior score = melhor atributo.*

In [19]:
inicio = time.time()

classes = np.unique(y_bin.values)

fisher_scores = []
for col in X_full.columns:
    overall_mean = X_full[col].mean()
    numerator = 0
    denominator = 0
    for c in classes:
        class_data = X_full[y_bin.values == c][col]
        class_mean = class_data.mean()
        class_var = class_data.var(ddof=1)
        numerator += len(class_data) * (class_mean - overall_mean) ** 2
        denominator += len(class_data) * class_var
    fisher_scores.append(numerator / (denominator + 1e-9))

ranking_df = montar_ranking(X_full.columns, fisher_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_FisherScore.xlsx", tempo_execucao,
               "Ranking Fisher Score salvo em 'Ranking_FisherScore.xlsx'")


✅ Ranking Fisher Score salvo em 'Ranking_FisherScore.xlsx'
⏱ Tempo de execução: 6.0411 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Gini Index\n\n*Menor índice = melhor atributo (mais pureza nas classes).*

In [20]:
inicio = time.time()

classes = np.unique(y_bin.values)

def gini_index(attribute, target):
    # CORREÇÃO: em coluna constante, qcut devolve tudo NaN; o laço abaixo não
    # visitava nenhum grupo e o Gini saía 0 ("pureza perfeita" = topo do ranking).
    # Agora NaN vira um grupo próprio (-1) e a impureza é calculada normalmente.
    bins = pd.Series(pd.qcut(attribute, q=10, duplicates='drop', labels=False)).fillna(-1).to_numpy()
    gini = 0
    for b in np.unique(bins):
        subset = target[bins == b]
        if len(subset) == 0:
            continue
        probs = np.bincount(subset, minlength=len(classes)) / len(subset)
        gini_bin = 1 - np.sum(probs ** 2)
        gini += (len(subset) / len(target)) * gini_bin
    return gini

gini_scores = [gini_index(X_full[col], y_bin.values) for col in X_full.columns]

ranking_df = montar_ranking(X_full.columns, gini_scores, ascending=True)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_GiniIndex.xlsx", tempo_execucao,
               "Ranking Índice de Gini salvo em 'Ranking_GiniIndex.xlsx'")


✅ Ranking Índice de Gini salvo em 'Ranking_GiniIndex.xlsx'
⏱ Tempo de execução: 0.6799 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Symmetrical Uncertainty\n\n*Maior valor = melhor atributo.*

In [21]:
from scipy.stats import entropy

inicio = time.time()

info_gain = mutual_info_classif(X_full, y_bin.values, discrete_features=False, random_state=42)  # determinístico

def entropy_discrete(values):
    probs = np.bincount(values) / len(values)
    return entropy(probs, base=2)

H_y = entropy_discrete(y_bin.values)

def entropy_continuous(column):
    bins = pd.qcut(column, q=10, duplicates='drop')
    probs = bins.value_counts(normalize=True)
    return entropy(probs, base=2)

H_x = [entropy_continuous(X_full[col]) for col in X_full.columns]
su_scores = [2 * ig / (hx + H_y + 1e-9) for ig, hx in zip(info_gain, H_x)]

ranking_df = montar_ranking(X_full.columns, su_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_SymmetricalUncertainty.xlsx", tempo_execucao,
               "Ranking Symmetrical Uncertainty salvo em 'Ranking_SymmetricalUncertainty.xlsx'")


✅ Ranking Symmetrical Uncertainty salvo em 'Ranking_SymmetricalUncertainty.xlsx'
⏱ Tempo de execução: 2.5668 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## MIC (Maximal Information Coefficient) — aproximado\n\n*Maior valor = melhor atributo.*

In [22]:
from sklearn.metrics import mutual_info_score

inicio = time.time()

def approximate_mic(x, y, bins=10):
    try:
        x_binned = pd.qcut(x, q=bins, duplicates='drop', labels=False).fillna(-1)
    except ValueError:
        x_binned = pd.cut(x, bins=bins, labels=False).fillna(-1)

    if len(np.unique(y)) > bins:
        try:
            y_binned = pd.qcut(y, q=bins, duplicates='drop', labels=False).fillna(-1)
        except ValueError:
            y_binned = pd.cut(y, bins=bins, labels=False).fillna(-1)
    else:
        y_binned = y

    mi = mutual_info_score(x_binned, y_binned)
    norm_factor = np.log2(min(len(np.unique(x_binned)), len(np.unique(y_binned))) + 1e-9)
    return mi / norm_factor if norm_factor > 0 else 0

mic_scores = [approximate_mic(X_full[col], y_bin.values) for col in X_full.columns]

ranking_df = montar_ranking(X_full.columns, mic_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_ApproxMIC.xlsx", tempo_execucao,
               "Ranking MIC (aproximado) salvo em 'Ranking_ApproxMIC.xlsx'")


✅ Ranking MIC (aproximado) salvo em 'Ranking_ApproxMIC.xlsx'
⏱ Tempo de execução: 1.0948 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Kolmogorov–Smirnov\n\n*Maior estatística KS = melhor atributo (distribuições mais distintas entre as classes; p-value não é mais salvo).*

In [23]:
from scipy.stats import ks_2samp

inicio = time.time()

classes = np.unique(y_bin.values)  # sempre [0, 1] com a base binária

ks_scores = []
for col in X_full.columns:
    group1 = X_full[y_bin.values == classes[0]][col]
    group2 = X_full[y_bin.values == classes[1]][col]
    ks_stat, p_val = ks_2samp(group1, group2)
    ks_scores.append(ks_stat)

ranking_df = montar_ranking(X_full.columns, ks_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_KS.xlsx", tempo_execucao,
               "Ranking Kolmogorov–Smirnov salvo em 'Ranking_KS.xlsx'")


✅ Ranking Kolmogorov–Smirnov salvo em 'Ranking_KS.xlsx'
⏱ Tempo de execução: 5.5263 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'


## Hellinger Distance\n\n*Maior distância = melhor atributo (distribuições mais separadas entre as classes).*

In [24]:
inicio = time.time()

classes = np.unique(y_bin.values)  # sempre [0, 1] com a base binária

def hellinger(p, q):
    return np.sqrt(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2)) / np.sqrt(2)

def compute_hellinger(feature, y, bins=10):
    group1 = feature[y == classes[0]]
    group2 = feature[y == classes[1]]

    hist1, _ = np.histogram(group1, bins=bins, range=(feature.min(), feature.max()), density=True)
    hist2, _ = np.histogram(group2, bins=bins, range=(feature.min(), feature.max()), density=True)

    p = hist1 / np.sum(hist1)
    q = hist2 / np.sum(hist2)
    return hellinger(p, q)

hellinger_scores = [compute_hellinger(X_full[col], y_bin.values) for col in X_full.columns]

ranking_df = montar_ranking(X_full.columns, hellinger_scores, ascending=False)

tempo_execucao = time.time() - inicio
salvar_ranking(ranking_df, "Ranking_HellingerDistance.xlsx", tempo_execucao,
               "Ranking Hellinger Distance salvo em 'Ranking_HellingerDistance.xlsx'")


✅ Ranking Hellinger Distance salvo em 'Ranking_HellingerDistance.xlsx'
⏱ Tempo de execução: 0.2155 segundos
📊 Tempo registrado em 'Tempos_Execucao.xlsx'
